# 06 - Iceberg lakehouse and JDBC integration - Solution


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("lesson06_iceberg_jdbc")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "hdfs://namenode:8020/warehouse")
    .getOrCreate()
)


In [ ]:
orders_df = spark.read.jdbc(
    url="jdbc:postgresql://postgres-tgt:5432/postgres",
    table="orders",
    properties={"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"},
)
curated_df = (
    orders_df.groupBy("dt")
    .agg(F.count("*").alias("rows_cnt"), F.round(F.sum("amount"), 2).alias("revenue"))
    .orderBy("dt")
)


In [ ]:
# Требует реального Iceberg catalog:
# curated_df.writeTo("local.db.sales_by_day").createOrReplace()
# spark.table("local.db.sales_by_day").show(20, truncate=False)
